# EasyRAG Context Assembly And Packing

## What you will do

- turn retrieved citations into a packed context block
- inspect the prompt produced by `build_generation_prompt()`
- run `generate_answer()` and compare the answer with its selected evidence

## Why this matters

The answer layer is easier to debug when you can see the packed context and final prompt directly. This notebook makes that prompt boundary explicit.


## Source code anchors

- `easyrag.rag.generation.packing.build_context_block`
- `easyrag.rag.generation.prompting.build_generation_prompt`
- `easyrag.rag.generation.pipeline.generate_answer`
- `easyrag.rag.types.AnswerParam`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.generation.packing import build_context_block
from easyrag.rag.generation.pipeline import generate_answer
from easyrag.rag.generation.prompting import build_generation_prompt
from easyrag.rag.generation.selection import select_answer_citations


## Deterministic path

We will retrieve evidence first, then make every generation-stage transformation visible: citation selection, context packing, prompt assembly, and the final grounded answer.


In [ ]:
packing_tmp = tempfile.TemporaryDirectory()
packing_rag = EasyRAG(
    working_dir=packing_tmp.name,
    workspace="packing-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(packing_rag.initialize_storages())
run_sync(
    packing_rag.ainsert(
        [
            "# Retrieval\nEasyRAG uses grounded retrieval and query rewrite to keep answers traceable.\n",
            "# Storage\nRetrieved evidence is packed before answer synthesis and storage review.\n",
            "# Policy\nCitation-aware answers expose the evidence budget directly to readers.\n",
        ],
        ids=["doc::retrieval", "doc::storage", "doc::policy"],
        file_paths=["docs/retrieval.md", "docs/storage.md", "docs/policy.md"],
    )
)
packing_query = "How does EasyRAG keep answers grounded?"
packing_result = run_sync(packing_rag.aquery(packing_query, QueryParam(mode="hybrid", chunk_top_k=5)))
packing_param = AnswerParam(max_citations=2, max_context_chars=180, style="citation_aware")
selected_citations = select_answer_citations(
    packing_result.citations,
    max_items=packing_param.max_citations,
    max_chars=packing_param.max_context_chars,
)
context_block = build_context_block(selected_citations)
prompt = build_generation_prompt(packing_query, context_block, packing_param)
packed_answer = generate_answer(
    packing_query,
    packing_result,
    answer_param=packing_param,
    answer_model_func=packing_rag.answer_model_func,
)

print("Selected citations")
_print_json(selected_citations)
print("\nContext block")
print(context_block)
print("\nPrompt")
print(prompt)
print("\nAnswer result")
_print_json({
    "answer": packed_answer.answer,
    "selected_citations": packed_answer.selected_citations,
    "metadata": packed_answer.metadata,
})


## Output inspection

The next cell changes the generation style and context budget so you can see which parts of the prompt change and which parts stay fixed.


In [ ]:
extractive_param = AnswerParam(max_citations=1, max_context_chars=110, style="extractive")
extractive_selected = select_answer_citations(
    packing_result.citations,
    max_items=extractive_param.max_citations,
    max_chars=extractive_param.max_context_chars,
)
extractive_context = build_context_block(extractive_selected)
extractive_prompt = build_generation_prompt(packing_query, extractive_context, extractive_param)
extractive_answer = generate_answer(
    packing_query,
    packing_result,
    answer_param=extractive_param,
    answer_model_func=packing_rag.answer_model_func,
)

print("Shorter extractive context")
print(extractive_context)
print("\nExtractive prompt header")
print("\n".join(extractive_prompt.splitlines()[:8]))
print("\nExtractive answer")
print(extractive_answer.answer)


## Provider-backed path

The optional path below uses provider-backed retrieval when available. Answer synthesis may still fall back to the built-in grounded answer helper unless you explicitly pass an `answer_model_func`.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(working_dir=provider_tmp.name, workspace="provider-packing-demo")
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                [
                    "# Retrieval\nGrounded retrieval feeds answer packing.\n",
                    "# Policy\nCitations keep the answer inspectable.\n",
                ],
                ids=["doc::retrieval", "doc::policy"],
                file_paths=["docs/retrieval.md", "docs/policy.md"],
            )
        )
        provider_answer = run_sync(
            provider_rag.aanswer(
                "How does EasyRAG keep answers inspectable?",
                QueryParam(mode="hybrid", chunk_top_k=3),
                AnswerParam(max_citations=2, max_context_chars=160),
            )
        )
        _print_json({"answer": provider_answer.answer, "metadata": provider_answer.metadata})
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()


## What changed and why

The generation stage keeps three layers distinct: selected citations, packed context, and final answer text. That separation is useful because prompt problems, selection problems, and synthesis problems are not the same bug.


In [ ]:
run_sync(packing_rag.finalize_storages())
packing_tmp.cleanup()
print("Cleaned up the deterministic packing workspace.")


## Next steps

- Continue with [06_03_retrieval_metrics.ipynb](../06_evaluation/06_03_retrieval_metrics.ipynb) to evaluate whether your retriever is good enough before you optimize generation further.
- Read [05-generation-overview.md](../../docs/05-generation-overview.md) for the chapter-level explanation of evidence selection, prompt packing, and answer synthesis.
- Compare this notebook with [05_02_evidence_selection_and_topk_for_answering.ipynb](05_02_evidence_selection_and_topk_for_answering.ipynb) if you want to isolate the selection step again.
